# Financial Fraud Detection using Machine Learning and XGBoost

In [ ]:
# Load the Dataset
from pathlib import Path
import pandas as pd

DATA_PATH = Path("data") / "PS_20174392719_1491204439457_log.csv"
df = pd.read_csv(DATA_PATH, nrows=100000)
df.head()

#### Data Understanding

In [ ]:
#Shape of the data
df.shape

In [ ]:
df.describe()

In [ ]:
#counts of Fraud vs No Fraud
df["isFraud"].value_counts()

#### EDA

In [ ]:
import matplotlib.pyplot as plt
df["isFraud"].value_counts().plot(kind="bar")
plt.title("Fraud vs Non-Fraud Count")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()

In [ ]:
#Checking null values
df.isna().sum()

In [ ]:
#Column names
df.columns

#### Data Cleaning

In [ ]:
#Dropping irrelevant columns
df = df.drop(["nameOrig", "nameDest"], axis=1, errors= "ignore")
df.head()

#### Feature Selection

In [ ]:
X = df.drop("isFraud", axis=1)
y = df["isFraud"]
#converting the "type" column
X_encoded = pd.get_dummies(X, columns=["type"], drop_first=True)

#### Splitting the training data and testing data

In [ ]:
from sklearn.model_selection import train_test_split

X_encoded_train, X_encoded_test, y_train, y_test = train_test_split(
    X_encoded, 
    y, 
    test_size=0.2,
    random_state=42,
    stratify = y
)

#### Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_encoded_train)
X_test_scaled = scaler.transform(X_encoded_test)

#### Baseline Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

#### Model Evaluation

In [ ]:
from sklearn.metrics import (
        confusion_matrix,
        accuracy_score,
        precision_score,
        recall_score,
        f1_score,
        classification_report
)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision Score:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Classification Report:")
print(classification_report(y_test, y_pred))

### Untuned XGBoost

In [ ]:
# Calculate imbalance ratio using training target only

non_fraud_count = y_train.value_counts()[0]
fraud_count = y_train.value_counts()[1]

scale_pos_weight = non_fraud_count / fraud_count

print("Non-fraud count:", non_fraud_count)
print("Fraud count:", fraud_count)
print("scale_pos_weight:", scale_pos_weight)

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)

xgb_model.fit(X_encoded_train, y_train)

y_pred_xgb = xgb_model.predict(X_encoded_test)

In [ ]:
print("XGBoost Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb))
print("Recall:", recall_score(y_test, y_pred_xgb))
print("F1 Score:", f1_score(y_test, y_pred_xgb))

print(classification_report(y_test, y_pred_xgb))

In [ ]:
xgb_cv_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)

In [ ]:
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_score

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

f1_scores = cross_val_score(
    xgb_cv_model,
    X_encoded_train,
    y_train,
    cv=skf,
    scoring="f1"
)

print("F1 scores:", f1_scores)
print("Mean F1:", f1_scores.mean())
print("Std F1:", f1_scores.std())

#### Recall Scores

In [ ]:
recall_scores = cross_val_score(
    xgb_cv_model,
    X_encoded_train,
    y_train,
    cv=skf,
    scoring="recall"
)

print("Recall scores:", recall_scores)
print("Mean Recall:", recall_scores.mean())
print("Std Recall:", recall_scores.std())

#### Precision Scores

In [ ]:
precision_scores = cross_val_score(
    xgb_cv_model,
    X_encoded_train,
    y_train,
    cv=skf,
    scoring="precision"
)

print("Precision scores:", precision_scores)
print("Mean Precision:", precision_scores.mean())
print("Std Precision:", precision_scores.std())

#### RandomizedSearchCV for XGBoost

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier

# WHY: Fraud imbalance ratio
non_fraud_count = y_train.value_counts()[0]
fraud_count = y_train.value_counts()[1]
scale_pos_weight = non_fraud_count / fraud_count

# WHY: Base XGBoost model
xgb_base = XGBClassifier(
    eval_metric="logloss",
    random_state=42
)

# WHY: Hyperparameter search space
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
    "scale_pos_weight": [scale_pos_weight]
}

# WHY: StratifiedKFold because fraud data is imbalanced
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# WHY: RandomizedSearchCV tries different parameter combinations
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_grid,
    n_iter=20,
    scoring="f1",
    cv=skf,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# WHY: Search best model
random_search.fit(X_encoded_train, y_train)

print("Best Parameters:")
print(random_search.best_params_)

print("Best CV F1 Score:")
print(random_search.best_score_)

In [ ]:
best_xgb_model = random_search.best_estimator_

y_pred_best_xgb = best_xgb_model.predict(X_encoded_test)

print("Best XGBoost Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_best_xgb))

print("Accuracy:", accuracy_score(y_test, y_pred_best_xgb))
print("Precision:", precision_score(y_test, y_pred_best_xgb))
print("Recall:", recall_score(y_test, y_pred_best_xgb))
print("F1 Score:", f1_score(y_test, y_pred_best_xgb))

print(classification_report(y_test, y_pred_best_xgb))

#### Predict Fraud Probabilities using Tuned XGBoost

In [ ]:
y_proba_best = best_xgb_model.predict_proba(X_encoded_test)[:, 1]

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

best_xgb_threshold_results = []

for threshold in thresholds:
    y_pred_threshold = (y_proba_best >= threshold).astype(int)

    cm = confusion_matrix(y_test, y_pred_threshold)
    tn, fp, fn, tp = cm.ravel()

    best_xgb_threshold_results.append({
        "threshold": threshold,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "precision": precision_score(y_test, y_pred_threshold),
        "recall": recall_score(y_test, y_pred_threshold),
        "f1_score": f1_score(y_test, y_pred_threshold)
    })

best_xgb_threshold_df = pd.DataFrame(best_xgb_threshold_results)
best_xgb_threshold_df

#### Final Threshold

In [ ]:
final_threshold = 0.4

y_pred_final = (y_proba_best >= final_threshold).astype(int)

print("Final Tuned XGBoost Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_final))

print("Final Precision:", precision_score(y_test, y_pred_final))
print("Final Recall:", recall_score(y_test, y_pred_final))
print("Final F1 Score:", f1_score(y_test, y_pred_final))

print(classification_report(y_test, y_pred_final))

### Final Results

In [ ]:
final_results = pd.DataFrame([
    {
        "Model": "Baseline Logistic Regression",
        "False Positives": 0,
        "False Negatives": 22,
        "True Positives": 1,
        "Precision": 1.00,
        "Recall": 0.043,
        "F1 Score": 0.083
    },
    {
        "Model": "Untuned XGBoost + Threshold 0.5",
        "False Positives": 85,
        "False Negatives": 2,
        "True Positives": 21,
        "Precision": 0.198,
        "Recall": 0.913,
        "F1 Score": 0.326
    },
    {
        "Model": "Tuned XGBoost + Threshold 0.4",
        "False Positives": 9,
        "False Negatives": 2,
        "True Positives": 21,
        "Precision": 0.700,
        "Recall": 0.913,
        "F1 Score": 0.792
    }
])

final_results

In [ ]:
# Optional: save model artifacts for deployment or reuse.
from pathlib import Path
import joblib

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(best_xgb_model, MODEL_DIR / "tuned_xgboost_fraud_model.pkl")
joblib.dump(list(X_encoded.columns), MODEL_DIR / "feature_columns.pkl")
joblib.dump(final_threshold, MODEL_DIR / "final_threshold.pkl")

## Final Conclusion

The final selected model is Tuned XGBoost with a probability threshold of 0.4.

This model achieved:
- Precision: 70.0%
- Recall: 91.3%
- F1-score: 79.2%
- False Positives: 9
- False Negatives: 2
- True Positives: 21

Since fraud detection is a recall-sensitive problem, the selected threshold provides a strong business tradeoff by detecting 21 out of 23 fraud cases while keeping false positives low.